# 🫁 MedGemma Chest X-Ray Analyzer

Analyze chest X-ray images using **Google's MedGemma 1.5 (4B)** medical vision-language model and generate structured radiology reports.

---

## Requirements
- ✅ **Colab T4 GPU** (Runtime → Change runtime type → T4 GPU)
- ✅ **HuggingFace token** (free) stored in Colab Secrets as `HF_TOKEN`
- ✅ **MedGemma license accepted** at [huggingface.co/google/medgemma-1.5-4b-it](https://huggingface.co/google/medgemma-1.5-4b-it)

---

⚠️ **Disclaimer**: For research purposes only. Not for clinical diagnosis.

In [ ]:
# CELL 0 — Fix CUDA Version Mismatch
# Colab sometimes ships PyTorch and torchvision compiled for different CUDA versions.
# That causes a RuntimeError when transformers tries to import torchvision internally.
# This cell detects the mismatch, reinstalls the matching torchvision, then restarts.
#
# HOW IT WORKS:
#   1. Run this cell first.
#   2. If a mismatch is found, the runtime restarts automatically.
#   3. After restart, run ALL cells again from the top — the mismatch will be gone.

import subprocess, sys, torch

torch_cuda = torch.version.cuda          # e.g. '13.0'
print(f'PyTorch version : {torch.__version__}')
print(f'PyTorch CUDA    : {torch_cuda}')

needs_restart = False
try:
    import torchvision
    from torchvision.transforms import functional  # triggers CUDA version check
    print(f'torchvision     : {torchvision.__version__} — CUDA versions match')
except RuntimeError as e:
    print(f'CUDA mismatch detected: {e}')
    needs_restart = True

if needs_restart:
    cuda_ver = torch_cuda.replace('.', '')   # '13.0' -> '130'
    print()
    print(f'Reinstalling torchvision for CUDA {torch_cuda} ...')
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q',
         'torchvision', '--force-reinstall',
         '--index-url', f'https://download.pytorch.org/whl/cu{cuda_ver}'],
        check=True
    )
    print()
    print('Runtime restarting to apply the fix...')
    print('After restart: run all cells again from the top.')
    import os
    os.kill(os.getpid(), 9)          # triggers Colab runtime restart
else:
    print('No fix needed — proceed to the next cell.')


In [ ]:
# CELL 1 — Install Required Packages
# transformers  -> loads and runs the MedGemma model
# accelerate    -> helps run the model efficiently on GPU
# bitsandbytes  -> enables memory-efficient model loading
# requests      -> used to download the sample X-ray image
# Pillow        -> image loading and processing library
#
# Note: torch and torchvision are pre-installed in Colab.
# We do NOT reinstall torch here — doing so risks a version conflict.
# The CUDA mismatch for torchvision was already fixed in Cell 0.

!pip install -q --upgrade transformers accelerate bitsandbytes requests "Pillow>=10.4.0"

print('All packages installed successfully')


In [ ]:
# CELL 2 — Load Your HuggingFace Token
# MedGemma is a gated model — you need a HuggingFace account and token to download it.
# This cell reads your token from Colab Secrets (the secure way to store passwords/keys).
#
# How to add your token to Colab Secrets:
#   1. Click the 🔑 key icon in the left sidebar
#   2. Click "Add new secret"
#   3. Name: HF_TOKEN  |  Value: your token from huggingface.co
#   4. Toggle "Notebook access" ON
#
# If Secrets aren't set up, paste your token directly into HF_TOKEN below as a fallback.

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    print('✅ HF_TOKEN loaded from Colab Secrets')
except Exception:
    HF_TOKEN = 'hf_your_token_here'  # Fallback: paste your token here
    print('⚠️  Colab Secrets not available — paste your token into HF_TOKEN above')

if HF_TOKEN and HF_TOKEN != 'hf_your_token_here':
    print(f'   Token loaded — first 8 chars: {HF_TOKEN[:8]}...')
else:
    print('❌ HF_TOKEN is not set — model download will fail')

In [ ]:
# CELL 3 — Check GPU
# This confirms that a GPU is active before we try to load the model.
# MedGemma is a 4 billion parameter model — it needs a GPU to run in reasonable time.
# On a T4 GPU (16GB VRAM), inference takes ~10 seconds per image.
# On CPU, the same task would take 30–90 minutes — essentially unusable.
#
# If you see ❌ No GPU: go to Runtime → Change runtime type → T4 GPU

import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU detected: {gpu_name}')
    print(f'   Available VRAM: {vram_gb:.1f} GB')
    if vram_gb < 6:
        print('⚠️  Less than 6GB VRAM — model may run out of memory')
else:
    print('❌ No GPU found — go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# CELL 4 — Download and Load the MedGemma Model
# This downloads Google's MedGemma 1.5 (4B) model from HuggingFace (~8 GB).
# The first time you run this it will take 2–4 minutes to download.
# After that, Colab caches it so it loads faster on subsequent runs in the same session.
#
# What each setting does:
#   image-text-to-text → tells the pipeline we're sending both an image and text
#   torch_dtype=bfloat16     → loads the model in half precision to save memory
#   device_map='auto'  → automatically places the model on the GPU

import time
from transformers import pipeline

MODEL_ID = 'google/medgemma-1.5-4b-it'

print(f'Loading {MODEL_ID}...')
print('First run downloads ~8 GB — this takes 2–4 minutes. Please wait...')
t0 = time.time()

pipe = pipeline(
    'image-text-to-text',
    model=MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    token=HF_TOKEN,
)

print(f'✅ Model ready — loaded in {time.time() - t0:.1f}s')

In [ ]:
# CELL 5 — Load a Sample Chest X-Ray
# Downloads a real chest X-ray from Wikimedia Commons (public domain, free to use).
# This gives us a test image to make sure the model is working before you upload your own.
# The image is displayed so you can see what it looks like before analysis.

import io
import requests
import matplotlib.pyplot as plt
from PIL import Image

SAMPLE_URL = 'https://upload.wikimedia.org/wikipedia/commons/c/c8/Chest_Xray_PA_3-8-2010.png'

print('Downloading sample chest X-ray...')
resp = requests.get(SAMPLE_URL, headers={'User-Agent': 'MedGemma-Colab/1.0'}, timeout=30)
resp.raise_for_status()
sample_image = Image.open(io.BytesIO(resp.content)).convert('RGB')
print(f'✅ Downloaded — image size: {sample_image.size[0]}x{sample_image.size[1]} pixels')

plt.figure(figsize=(8, 8))
plt.imshow(sample_image, cmap='gray')
plt.title('Sample Chest X-Ray (PA View — Wikimedia Commons, CC0)', fontsize=12)
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# CELL 6 — Analysis 1: Detailed Radiology Report
# Sends the sample X-ray to MedGemma and asks for a full structured report.
# The report covers all major sections a radiologist would include:
# Lungs, Pleura, Heart, Mediastinum, Bones, and a final Impression.
# This takes about 10 seconds on a T4 GPU.

from IPython.display import Markdown, display

SYSTEM_PROMPT = (
    'You are an expert radiologist. Analyze this chest X-ray and produce a '
    'structured report covering: Lungs, Pleura, Heart, Mediastinum, Bones, '
    'and a final Impression.'
)

messages = [
    {
        'role': 'system',
        'content': [{'type': 'text', 'text': SYSTEM_PROMPT}]
    },
    {
        'role': 'user',
        'content': [
            {'type': 'image', 'image': sample_image},
            {'type': 'text', 'text': 'Generate a detailed radiology report for this chest X-ray.'},
        ],
    },
]

print('Generating detailed radiology report...')
t0 = time.time()

output = pipe(text=messages, max_new_tokens=1024, do_sample=False)
# Extract just the model's reply from the full conversation output
report = output[0]['generated_text'][-1]['content']

print(f'Done in {time.time() - t0:.1f}s\n')
display(Markdown(f'## 📋 Detailed Radiology Report\n\n{report}'))
display(Markdown('---\n> ⚠️ *For research purposes only. Not for clinical diagnosis.*'))

In [ ]:
# CELL 7 — Key Findings + Patient-Friendly Explanation
# This cell defines a reusable analyze_xray() function and runs two more analyses.
#
# Key Findings → asks for a quick bullet-point summary instead of a full report.
#               Useful when you just want to know what's notable at a glance.
#
# Patient-Friendly → switches the system role from "radiologist" to "doctor explaining
#               to a patient". The output uses plain language, no medical jargon.
#               Great for sharing results with non-medical audiences.

from PIL import Image

def analyze_xray(img, prompt, system_prompt=None):
    # Use the radiologist prompt by default unless a custom role is passed in
    sys_content = system_prompt or SYSTEM_PROMPT

    # Build the message in chat format: system role + image + question
    msgs = [
        {'role': 'system', 'content': [{'type': 'text', 'text': sys_content}]},
        {'role': 'user',   'content': [
            {'type': 'image', 'image': img},
            {'type': 'text',  'text': prompt},
        ]},
    ]
    result = pipe(text=msgs, max_new_tokens=1024, do_sample=False)
    # Extract just the model's reply from the full output
    return result[0]['generated_text'][-1]['content']


# Analysis 2: Key Findings as bullet points
print('=== Extracting Key Findings ===')
findings = analyze_xray(
    sample_image,
    'List the key findings in this chest X-ray as bullet points.'
)
display(Markdown('## 📌 Key Findings\n\n' + findings))


# Analysis 3: Plain language explanation for patients
print('\n=== Generating Patient-Friendly Explanation ===')
simple_explanation = analyze_xray(
    sample_image,
    'Explain what you see in this X-ray in simple terms a patient could understand.',
    system_prompt=(
        'You are a compassionate doctor explaining X-ray results to a patient with no '
        'medical background. Use plain, clear language.'
    )
)
display(Markdown('## 💬 Patient-Friendly Explanation\n\n' + simple_explanation))

In [ ]:
# CELL 8 — Analyze YOUR OWN Medical Image
# ─────────────────────────────────────────────────────────────────────────
# STEP 1: SET IMAGE TYPE
#
# Option A — Auto-detect (recommended): set IMAGE_TYPE = 'auto'
#   MedGemma will first classify what kind of image you uploaded,
#   then automatically switch to the right specialist prompt.
#   Takes ~5 extra seconds but removes the need to manually set the type.
#
# Option B — Manual: uncomment the line that matches your image.
#   Use this when auto-detect gives a wrong result.

IMAGE_TYPE = 'auto'             # Auto-detect (MedGemma identifies the modality)
# IMAGE_TYPE = 'chest_xray'     # Chest X-ray       — 88.9% macro F1 (MIMIC)
# IMAGE_TYPE = 'ct_scan'        # CT scan slice      — 61.1% accuracy
# IMAGE_TYPE = 'mri_brain'      # Brain MRI          — neuroradiologist prompt
# IMAGE_TYPE = 'mri_other'      # Other MRI (spine/abdomen/pelvis)
# IMAGE_TYPE = 'mammogram'      # Mammogram          — BI-RADS assessment
# IMAGE_TYPE = 'skin_lesion'    # Skin lesion        — ABCDE criteria
# IMAGE_TYPE = 'pathology_slide'# Histopathology     — cell morphology
# IMAGE_TYPE = 'fundus'         # Retinal fundus     — optic disc/macula
# IMAGE_TYPE = 'ultrasound'     # Ultrasound         — echogenicity/organs
# IMAGE_TYPE = 'bone_xray'      # Bone/joint X-ray   — fractures/lesions

# ─────────────────────────────────────────────────────────────────────────
# Specialist system prompts — one per modality.
# MedGemma uses this to know WHICH doctor it is and what anatomy to report on.
SYSTEM_PROMPTS = {
    'chest_xray':      'You are an expert chest radiologist with 20 years of experience. Analyze the chest X-ray and produce a structured report covering Lungs, Pleura, Heart, Mediastinum, Bones, and Impression.',
    'ct_scan':         'You are an expert radiologist specializing in CT interpretation. Analyze this CT scan slice, describe visible structures and density values, and provide a clinical impression.',
    'mri_brain':       'You are an expert neuroradiologist with 20 years of experience. Analyze this brain MRI. Evaluate brain parenchyma, ventricles, white matter, posterior fossa, any masses or signal abnormalities, midline shift, and provide a structured impression.',
    'mri_other':       'You are an expert MRI radiologist. Analyze this MRI, describe anatomical structures, signal intensities, any abnormal regions, and provide a clinical impression.',
    'mammogram':       'You are an expert breast radiologist. Analyze this mammogram using BI-RADS criteria. Describe breast composition, masses, calcifications, and assign a BI-RADS category.',
    'skin_lesion':     'You are an expert dermatologist. Evaluate this skin lesion using ABCDE criteria: Asymmetry, Border, Color, Diameter, Evolution. Provide a ranked differential diagnosis.',
    'pathology_slide': 'You are an expert pathologist. Analyze this histopathology slide. Describe cell morphology, nuclear features, tissue architecture, and provide a diagnostic impression.',
    'fundus':          'You are an expert ophthalmologist. Analyze this fundus photograph. Describe the optic disc, macula, retinal vasculature, and any pathological findings.',
    'ultrasound':      'You are an expert ultrasound radiologist. Analyze this ultrasound. Describe the organ(s), echogenicity, any masses or fluid, and provide a clinical impression.',
    'bone_xray':       'You are an expert musculoskeletal radiologist. Analyze this bone X-ray. Describe cortical integrity, bone density, joint spaces, any fractures or lesions, and provide a clinical impression.',
}

# Per-modality detailed report prompts — which anatomical sections to cover.
DETAILED_PROMPTS = {
    'chest_xray':      'Generate a detailed radiology report covering: 1. Lungs  2. Pleura  3. Heart  4. Mediastinum  5. Bones  6. Impression.',
    'ct_scan':         'Analyze this CT scan. Describe visible structures, attenuation values, any masses or lesions, and provide a clinical impression.',
    'mri_brain':       'Generate a detailed brain MRI report covering: 1. Brain parenchyma  2. Ventricles and CSF  3. White matter signal  4. Posterior fossa  5. Masses or signal abnormalities  6. Midline shift  7. Impression.',
    'mri_other':       'Analyze this MRI. Describe structures, signal intensities, any abnormal regions, and provide a clinical impression.',
    'mammogram':       'Analyze this mammogram. Describe breast composition, masses, calcifications, architectural distortion, and provide a BI-RADS category.',
    'skin_lesion':     'Evaluate using ABCDE criteria (Asymmetry, Border, Color, Diameter, Evolution). Describe morphology and provide a ranked differential diagnosis.',
    'pathology_slide': 'Analyze this slide. Describe cell morphology, nuclear features, tissue architecture, staining patterns, and provide a diagnostic impression.',
    'fundus':          'Analyze this fundus. Describe the optic disc, macula, retinal vasculature, and any pathological findings.',
    'ultrasound':      'Analyze this ultrasound. Describe organ(s), echogenicity, borders, any masses or fluid, and provide a clinical impression.',
    'bone_xray':       'Analyze this bone X-ray. Describe cortical integrity, bone density, joint spaces, fractures, and provide a clinical impression.',
}

# Short classification prompt used by auto-detect.
# We cap the response at 30 tokens — only a single label word is needed.
AUTO_DETECT_PROMPT = (
    'What type of medical image is this? '
    'Reply with exactly one label from: '
    'chest_xray, ct_scan, mri_brain, mri_other, mammogram, '
    'skin_lesion, pathology_slide, fundus, ultrasound, bone_xray. '
    'Output only the label, nothing else.'
)

def auto_detect_image_type(img):
    """Ask MedGemma to classify the image modality in ~5s.
    Returns the matched image_type key or 'chest_xray' as fallback."""
    known_types = list(SYSTEM_PROMPTS.keys())
    classification_messages = [
        {'role': 'system',
         'content': [{'type': 'text', 'text': 'You are a medical imaging classifier. Reply with exactly one label from the list provided.'}]},
        {'role': 'user',
         'content': [
             {'type': 'image', 'image': img},
             {'type': 'text',  'text': AUTO_DETECT_PROMPT},
         ]},
    ]
    # Use only 30 tokens — we only need one short label back
    result = pipe(text=classification_messages, max_new_tokens=30, do_sample=False)
    response = result[0]['generated_text'][-1]['content'].strip().lower()
    print('Classification response:', response)
    for key in known_types:
        if key in response:
            return key
    return 'chest_xray'  # safe fallback

# ─────────────────────────────────────────────────────────────────────────
# STEP 2: UPLOAD YOUR IMAGE

import io, os
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import Markdown, display
from google.colab import files

print('Select your image file (PNG or JPG):')
uploaded = files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]
    custom_image = Image.open(io.BytesIO(uploaded[filename])).convert('RGB')
    print('Uploaded :', filename)
    print('Size     :', custom_image.size[0], 'x', custom_image.size[1], 'px')

    plt.figure(figsize=(7, 7))
    plt.imshow(custom_image, cmap='gray')
    plt.title(filename, fontsize=11)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    # ── Resolve actual image type ─────────────────────────────────────────
    if IMAGE_TYPE == 'auto':
        print('Auto-detecting image type...')
        resolved_type = auto_detect_image_type(custom_image)
        print('Detected type :', resolved_type)
    else:
        # Validate manual selection — catch typos before running inference
        if IMAGE_TYPE not in SYSTEM_PROMPTS:
            raise ValueError('Unknown IMAGE_TYPE: "' + IMAGE_TYPE + '". Valid: ' + str(list(SYSTEM_PROMPTS.keys())))
        resolved_type = IMAGE_TYPE
        print('Image type :', resolved_type)

    print('Specialist :', SYSTEM_PROMPTS[resolved_type][:80], '...')
    print()

    # ── Run analysis using the resolved modality prompts ──────────────────
    # analyze_xray() is defined in Cell 6. We override the default chest
    # system prompt with the one matched to resolved_type so MedGemma acts
    # as the correct specialist for this image.
    print('Generating report (~10 s on T4 GPU)...')
    custom_report = analyze_xray(
        custom_image,
        DETAILED_PROMPTS[resolved_type],
        system_prompt=SYSTEM_PROMPTS[resolved_type],
    )

    display(Markdown('## Report: ' + filename + '  [' + resolved_type + ']\n\n' + custom_report))
    display(Markdown('---\n> For research purposes only. Not for clinical diagnosis.'))
else:
    print('No file uploaded — run this cell again and select a file.')
